In [11]:
import pandas as pd
import os
import json
import subprocess

In [6]:
# run this once
keyword = '_liver_TEST_500bp.bed'
test_dir = '/home/azstephe/liverRegression/regression_liver/data/test_splits/'

out_dir = '/home/azstephe/liverRegression/regression_liver/data/alphagenome_inputs/'

stats_log = []
summary_output_path = "/home/azstephe/liverRegression/regression_liver/data/alphagenome_preprocessing_summary.tsv"

for d in ['log_pos', 'neg', 'log_LiuAll_test1', 'log_test2', 'log_test3']:
    dr = os.path.join(test_dir, d)
    
    if not os.path.exists(dr):
        continue

    for file in os.listdir(dr):
        if "newRat" in file:
            continue
        
        filepath = os.path.join(dr, file)
        
        # Filtering logic
        if 'non' not in filepath and keyword in filepath:
            # Extract species (e.g., "mouse" from "mouse_liver_peaks.bed")
            species = file.split('_')[0]
            
            cols = ['chrom', 'start', 'end', 'peak', 'signal']
            df = pd.read_csv(filepath, sep='\t', names=cols, index_col=False)
            
            extension = 1048576 / 2
            df['new_start'] = (df['start'] - extension + 250).astype(int)
            df['new_end'] = (df['end'] + extension - 250).astype(int)

            # Load chromosome sizes
            chrom_size_path = f'/home/azstephe/liverRegression/regression_liver/data/chrom_sizes/{species}_chrom_sizes.json'
            with open(chrom_size_path, 'r') as f:
                chrom_map = json.load(f)

            df['max_len'] = df['chrom'].map(chrom_map)

            # Filter peaks that fall off the chromosome boundaries
            valid_sections = df[(df['new_start'] >= 0) & (df['new_end'] <= df['max_len'])].copy()

            # --- COLLECT STATS ---
            num_orig = len(df)
            num_kept = len(valid_sections)
            num_lost = num_orig - num_kept
            
            stats_log.append({
                'directory': d,
                'species': species,
                'filename': file,
                'original_peaks': num_orig,
                'peaks_kept': num_kept,
                'peaks_lost': num_lost,
                'loss_percentage': round((num_lost / num_orig) * 100, 2) if num_orig > 0 else 0
            })
            # ---------------------

            # # Save the filtered BED file
            out_dr = os.path.join(out_dir, d)
            os.makedirs(out_dr, exist_ok=True)

            outfile = os.path.join(out_dr, file.replace("_500bp", "_1Mb"))
            
            valid_sections[['chrom', 'new_start', 'new_end', 'peak', 'signal']].to_csv(
                outfile, 
                sep='\t', 
                index=False, 
                header=False
            )

# Final step: Save the summary table
stats_df = pd.DataFrame(stats_log)
stats_df.to_csv(summary_output_path, sep='\t', index=False)
print(f"Preprocessing complete. Summary saved to: {summary_output_path}")

Preprocessing complete. Summary saved to: /home/azstephe/liverRegression/regression_liver/data/alphagenome_preprocessing_summary.tsv


In [7]:
stats_df

,directory,species,filename,original_peaks,peaks_kept,peaks_lost,loss_percentage
0,log_pos,cow,cow_liver_TEST_500bp.bed,1664,1646,18,1.08
1,log_pos,pig,pig_liver_TEST_500bp.bed,1627,1619,8,0.49
2,log_pos,rat,rat_liver_TEST_500bp.bed,2576,2562,14,0.54
3,log_pos,mouse,mouse_liver_TEST_500bp.bed,3776,3760,16,0.42
4,log_pos,macaque,macaque_liver_TEST_500bp.bed,2624,2616,8,0.30
5,neg,pig,pig_liver_TEST_500bp.bed,3057,2846,211,6.90
6,neg,rat,rat_liver_TEST_500bp.bed,4405,4395,10,0.23
7,neg,macaque,macaque_liver_TEST_500bp.bed,3477,3463,14,0.40
8,neg,cow,cow_liver_TEST_500bp.bed,3720,2959,761,20.46
9,neg,mouse,mouse_liver_TEST_500bp.bed,3698,3689,9,0.24


In [9]:
stats_df = pd.read_csv("/home/azstephe/liverRegression/regression_liver/data/alphagenome_preprocessing_summary.tsv", sep='\t')
stats_df

,directory,species,filename,original_peaks,peaks_kept,peaks_lost,loss_percentage
0,log_pos,cow,cow_liver_TEST_500bp.bed,1664,1646,18,1.08
1,log_pos,pig,pig_liver_TEST_500bp.bed,1627,1619,8,0.49
2,log_pos,rat,rat_liver_TEST_500bp.bed,2576,2562,14,0.54
3,log_pos,mouse,mouse_liver_TEST_500bp.bed,3776,3760,16,0.42
4,log_pos,macaque,macaque_liver_TEST_500bp.bed,2624,2616,8,0.30
5,neg,pig,pig_liver_TEST_500bp.bed,3057,2846,211,6.90
6,neg,rat,rat_liver_TEST_500bp.bed,4405,4395,10,0.23
7,neg,macaque,macaque_liver_TEST_500bp.bed,3477,3463,14,0.40
8,neg,cow,cow_liver_TEST_500bp.bed,3720,2959,761,20.46
9,neg,mouse,mouse_liver_TEST_500bp.bed,3698,3689,9,0.24


In [12]:
base_dir = '/home/azstephe/liverRegression/regression_liver/data/alphagenome_inputs/'
sub_dirs = ['log_pos', 'neg', 'log_LiuAll_test1', 'log_test2', 'log_test3']
# sub_dirs = []

genomes = {
    'cow': '/home/azstephe/regression_liver/data/splits/cowMouse/GCF_000003205.7_Btau_5.0.1_genomic_chromName.fna',
    'rat': '/projects/pfenninggroup/machineLearningForComputationalBiology/regElEvoGrant/RatGenome/rn6.fa',
    'macaque': '/projects/pfenninggroup/machineLearningForComputationalBiology/regElEvoGrant/MacaqueGenome/rheMac8.fa',
    'mouse': '/projects/pfenninggroup/machineLearningForComputationalBiology/regElEvoGrant/MouseGenome/mm10.fa',
    'pig': '/data/pfenninggroup/machineLearningForComputationalBiology/regElEvoGrant/200MammalsFastas/susScr3.fa'
}

for d in sub_dirs:
    dr = os.path.join(base_dir, d)
    
    if not os.path.exists(dr):
        print(f"Directory not found, skipping: {dr}")
        continue

    print(f"\nProcessing directory: {d}")
    
    for file in os.listdir(dr):
        # Only process BED or narrowPeak files
        if file.endswith('.bed') or file.endswith('.narrowPeak'):
            input_path = os.path.join(dr, file)
            species = file.split('_')[0]
            
            # Create output filename by replacing extension with .fa
            output_file = os.path.splitext(file)[0] + ".fa"
            output_path = os.path.join(dr, output_file)
            
            print(f"{file} -> {output_file}")

            genome_fasta = genomes[species]
            # Construct the command
            cmd = [
                "bedtools", "getfasta",
                "-fi", genome_fasta,
                "-bed", input_path,
                "-fo", output_path
            ]
            
            #Run the command
            try:
                subprocess.run(cmd, check=True, capture_output=True, text=True)
            except subprocess.CalledProcessError as e:
                print(f"  Error processing {file}: {e.stderr}")


Processing directory: log_pos
mouse_liver_TEST_1Mb.bed -> mouse_liver_TEST_1Mb.fa
macaque_liver_TEST_1Mb.bed -> macaque_liver_TEST_1Mb.fa
cow_liver_TEST_1Mb.bed -> cow_liver_TEST_1Mb.fa
pig_liver_TEST_1Mb.bed -> pig_liver_TEST_1Mb.fa
rat_liver_TEST_1Mb.bed -> rat_liver_TEST_1Mb.fa

Processing directory: neg
pig_liver_TEST_1Mb.bed -> pig_liver_TEST_1Mb.fa
mouse_liver_TEST_1Mb.bed -> mouse_liver_TEST_1Mb.fa
rat_liver_TEST_1Mb.bed -> rat_liver_TEST_1Mb.fa
macaque_liver_TEST_1Mb.bed -> macaque_liver_TEST_1Mb.fa
cow_liver_TEST_1Mb.bed -> cow_liver_TEST_1Mb.fa

Processing directory: log_LiuAll_test1
macaque_liver_TEST_1Mb.bed -> macaque_liver_TEST_1Mb.fa
cow_liver_TEST_1Mb.bed -> cow_liver_TEST_1Mb.fa
pig_liver_TEST_1Mb.bed -> pig_liver_TEST_1Mb.fa
rat_liver_TEST_1Mb.bed -> rat_liver_TEST_1Mb.fa

Processing directory: log_test2
macaque_liver_TEST_1Mb.bed -> macaque_liver_TEST_1Mb.fa
cow_liver_TEST_1Mb.bed -> cow_liver_TEST_1Mb.fa
rat_liver_TEST_1Mb.bed -> rat_liver_TEST_1Mb.fa
pig_liver_TES